# ENLIGHT Account Validation - Rule by Rule Comparison

Find EXACTLY which CA/BP is missing. Compare program export vs validation counts.

In [ ]:
import pandas as pd
import os
from collections import defaultdict

# Load program CSV
program_csv_path = "/tmp/CA_BP_rules.csv"  # <-- UPDATE THIS PATH

df = pd.read_csv(program_csv_path, header=None, names=['Rule', 'Value'])
print(f"Loaded {len(df)} records from program CSV")

# Group by rule
rule_groups = df.groupby('Rule')['Value'].apply(set).to_dict()

# Summary
print("\n" + "="*60)
print("PROGRAM CSV - ALL RULES SUMMARY")
print("="*60)
for rule in sorted(rule_groups.keys()):
    print(f"{rule:<25} : {len(rule_groups[rule]):>8} records")

## Step 1: Enter Validation Counts

Enter the counts from your ABAP validation report here.

In [ ]:
# Enter validation counts from ABAP report
validation_counts = {
    'Rule1: Vkont':    150000,  # <-- Enter from ABAP report
    'Rule1: Partner': 145000,  # <-- Enter from ABAP report
    'Rule2: Vkont':    25000,  # <-- Enter from ABAP report
    'Rule2: Partner': 24000,  # <-- Enter from ABAP report
    'Rule3: Vkont':     5000,  # <-- Enter from ABAP report
    'Rule4: Vkont':    80000,  # <-- Enter from ABAP report
}

print("Entered validation counts:")
for k, v in validation_counts.items():
    print(f"  {k}: {v}")

## Step 2: Compare Rule by Rule

Shows difference between validation and program for each rule.

In [ ]:
print("\n" + "="*70)
print("RULE-BY-RULE COMPARISON: Validation vs Program")
print("="*70)
print(f"{'Rule':<25} {'Validation':>12} {'Program':>12} {'Diff':>10} {'Status':<8}")
print("-"*70)

differences = {}

for rule, prog_count in sorted(rule_groups.items()):
    val_count = validation_counts.get(rule, 0)
    prog_val = len(prog_count)
    diff = val_count - prog_val
    
    differences[rule] = {
        'validation': val_count,
        'program': prog_val,
        'diff': diff
    }
    
    status = "✅" if diff == 0 else "⚠️"
    diff_str = f"{diff:+" if diff >= 0 else str(diff)
    print(f"{rule:<25} {val_count:>12} {prog_val:>12} {diff_str:>10} {status}")

print("-"*70)

# Highlight rules with differences
issues = {k: v for k, v in differences.items() if v['diff'] != 0}
if issues:
    print(f"\n⚠️  Found {len(issues)} rules with differences!")
    for rule, data in issues.items():
        print(f"   {rule}: Validation={data['validation']}, Program={data['program']}, Diff={data['diff']}")

## Step 3: Find Missing CA/BP for Specific Rule

Enter the rule name to see which CA/BP is missing (validation - program).

In [ ]:
# Enter the rule you want to investigate
target_rule = "Rule1: Vkont"  # <-- CHANGE THIS to the rule with difference

if target_rule not in rule_groups:
    print(f"Rule '{target_rule}' not found in program CSV")
    print(f"Available rules: {list(rule_groups.keys())}")
else:
    print(f"\n{'='*70}")
    print(f"ANALYZING: {target_rule}")
    print(f"{'='*70}")
    
    # Get program values
    program_values = rule_groups[target_rule]
    
    print(f"\nProgram CSV has {len(program_values)} unique values")
    
    # If you have validation export, load it here
    # validation_export_path = "/tmp/validation_rule1.csv"
    # validation_values = set(pd.read_csv(validation_export_path, header=None)[0])
    
    # Show sample of program values
    print(f"\nSample values from program (first 20):")
    for v in sorted(program_values)[:20]:
        print(f"  {v}")

## Step 4: Find Specific CA/BP

Search for a specific CA or BP number to see which rule it belongs to.

In [ ]:
# Enter CA/BP to search
search_value = "1234567890"  # <-- CHANGE THIS

print(f"Searching for: {search_value}")
print("="*50)

found_in = []
for rule, values in sorted(rule_groups.items()):
    if search_value in values:
        found_in.append(rule)
        print(f"✅ Found in: {rule}")

if not found_in:
    print(f"❌ NOT FOUND in any rule!")
    print("This CA/BP is missing from program export.")

## Step 5: List All Values for a Rule

Export all CA/BP for a specific rule to investigate further.

In [ ]:
# Export rule to CSV
export_rule = "Rule1: Vkont"  # <-- CHANGE THIS
export_path = "/tmp/rule1_vkont_export.csv"

if export_rule in rule_groups:
    export_df = pd.DataFrame({'VKONT': sorted(rule_groups[export_rule])})
    export_df.to_csv(export_path, index=False, header=False)
    print(f"Exported {len(export_df)} values to: {export_path}")
    print(f"\nFirst 10 values:")
    print(export_df.head(10).to_string(index=False))
else:
    print(f"Rule '{export_rule}' not found")

---

## Summary

This notebook helps you:
1. Compare validation counts vs program counts by rule
2. Identify which rules have differences
3. Search for specific CA/BP to find missing ones
4. Export values for further analysis

**Next step:** If you found a missing CA, check the ABAP logic for that rule.